# DTC Data Preparation with Feature Importance

Here we prepare the data for the Decision Tree Classifier model training.
We apply feature importance dimensionality reduction on DCT classifiers by 
making an initial tree which we can use to evaluate the most relevant 
features that affect the Tree decision classefier model 


1. Load the existing train/test split if available.
2. Trains an initial Decision Tree on the training data only.
4. Uses the initial tree's feature importance to select the top features.
5. Saves the reduced datasets into:

    ```text
    DTC/
    ├── train/
    │   ├── x_train_dtc_selected.csv
    │   └── y_train.csv
    └── test/
        ├── X_test_dtc_selected.csv
        └── y_test.csv
    ```

The initial tree is only used for feature selection. The final Decision Tree will be trained separately.

In [18]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

## Configuration

In [19]:
CLEANED_DATA_PATH = "cleaned.csv"

BASE_TRAIN_DIR = "train"
BASE_TEST_DIR = "test"

DTC_DIR = "DTC"
DTC_TRAIN_DIR = os.path.join(DTC_DIR, "train")
DTC_TEST_DIR = os.path.join(DTC_DIR, "test")

os.makedirs(DTC_TRAIN_DIR, exist_ok=True)
os.makedirs(DTC_TEST_DIR, exist_ok=True)

RANDOM_STATE = 42
TEST_SIZE = 0.30
TOP_FEATURE_COUNT = 15

## Load the split

In [20]:
x_train = pd.read_csv(os.path.join(BASE_TRAIN_DIR, "x_train.csv"))
x_test = pd.read_csv(os.path.join(BASE_TEST_DIR, "X_test.csv"))
y_train = pd.read_csv(os.path.join(BASE_TRAIN_DIR, "y_train.csv")).squeeze("columns")
y_test = pd.read_csv(os.path.join(BASE_TEST_DIR, "y_test.csv")).squeeze("columns")
print("Loaded existing train/test split.")


print("x_train shape:", x_train.shape)
print("x_test shape:", x_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

Loaded existing train/test split.
x_train shape: (4930, 22)
x_test shape: (2113, 22)
y_train shape: (4930,)
y_test shape: (2113,)


## Train initial Decision Tree for feature importance

This tree is only used to rank features. It is not the final model.

In [21]:
initial_tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
initial_tree.fit(x_train, y_train)

feature_importance = pd.DataFrame({
    "Feature": x_train.columns,
    "Importance": initial_tree.feature_importances_
}).sort_values(by="Importance", ascending=False)

feature_importance.to_csv(os.path.join(DTC_DIR, "initial_tree_feature_importance.csv"), index=False)

feature_importance

,Feature,Importance
3,TotalCharges,0.225548
1,tenure,0.222582
2,MonthlyCharges,0.199690
6,InternetService_Fiber optic,0.104974
18,PaperlessBilling_Yes,0.030681
20,PaymentMethod_Electronic check,0.030441
4,Partner_Yes,0.022043
0,SeniorCitizen,0.019749
5,Dependents_Yes,0.019641
9,OnlineSecurity_Yes,0.018833


## Select top features


We selected the top 15 features based on the previous table
hence the, `TOP_FEATURE_COUNT = 15`

We could go for lower but we will stick to this for now

In [22]:
selected_features = feature_importance.head(TOP_FEATURE_COUNT)["Feature"].tolist()

selected_features_df = pd.DataFrame({"Selected Feature": selected_features})
selected_features_df.to_csv(os.path.join(DTC_DIR, "selected_features.csv"), index=False)

x_train_dtc_selected = x_train[selected_features]
x_test_dtc_selected = x_test[selected_features]

print("Original number of features:", x_train.shape[1])
print("Selected number of features:", x_train_dtc_selected.shape[1])
print("Selected features:")
for feature in selected_features:
    print("-", feature)

Original number of features: 22
Selected number of features: 15
Selected features:
- TotalCharges
- tenure
- MonthlyCharges
- InternetService_Fiber optic
- PaperlessBilling_Yes
- PaymentMethod_Electronic check
- Partner_Yes
- SeniorCitizen
- Dependents_Yes
- OnlineSecurity_Yes
- TechSupport_Yes
- PaymentMethod_Mailed check
- Contract_One year
- StreamingMovies_No internet service
- StreamingMovies_Yes


## Save DTC-ready datasets

In [23]:
x_train_dtc_selected.to_csv(os.path.join(DTC_TRAIN_DIR, "x_train_dtc_selected.csv"), index=False)
x_test_dtc_selected.to_csv(os.path.join(DTC_TEST_DIR, "x_test_dtc_selected.csv"), index=False)

y_train.to_csv(os.path.join(DTC_TRAIN_DIR, "y_train.csv"), index=False, header=["Churn"])
y_test.to_csv(os.path.join(DTC_TEST_DIR, "y_test.csv"), index=False, header=["Churn"])

print("DTC feature-selected datasets saved successfully.")
print("Saved files:")
print(os.path.join(DTC_TRAIN_DIR, "x_train_dtc_selected.csv"))
print(os.path.join(DTC_TRAIN_DIR, "y_train.csv"))
print(os.path.join(DTC_TEST_DIR, "x_test_dtc_selected.csv"))
print(os.path.join(DTC_TEST_DIR, "y_test.csv"))

DTC feature-selected datasets saved successfully.
Saved files:
DTC\train\x_train_dtc_selected.csv
DTC\train\y_train.csv
DTC\test\x_test_dtc_selected.csv
DTC\test\y_test.csv
